
# ModernBERT Binary SFT: Industrial Guardrail
### A Specialized Binary Classification Approach for Query Filtering

This notebook demonstrates the **Supervised Fine-Tuning (SFT)** of a **ModernBERT** model as an ultra-efficient **Guardrail**. It specifically implements **Stage 4** of the industrial-grade pipeline to filter user queries into **ALLOW** (Relevant) vs. **BLOCK** (Off-topic/Irrelevant).

The goal is to adapt **ModernBERT**—a state-of-the-art encoder model—to act as a high-performance gatekeeper, ensuring that only specialized technical queries reach the expensive LLM layer. This process involves:
- Implementing the **Real-world Distribution Alignment** strategy for data generation.
- Fine-tuning the model using **BF16** and **Fused AdamW** on A100 hardware.
- Establishing a **Traffic Light Logic** for real-time inference filtering.
- Monitoring training stability using custom **Guardrail Surveillance** callbacks.

**Reference**: [Fine-tuning ModernBERT as an Efficient Guardrail for LLMs](https://medium.com/pythoneers/fine-tuning-modernbert-as-an-efficient-guardrail-for-llms-c0016cc83350)

In [ ]:
import os
import time
import torch
import numpy as np
from loguru import logger
from datasets import load_dataset
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    pipeline,
)

# Force GPU 0 and Mirror
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
logger.info("✅ Environment Setup Complete")

## Step1.  Data Generation

This is a deep data generation scheme designed to eliminate the "Synthetic-to-Real Distribution Gap." The core philosophy: **Real Seeds as Genotype, Logical Models as Skeleton, and Noise Engineering as Skin.**

### 1.1 Genotype Extraction (Real-world Seed Extraction)
**Goal**: Acquire "Noise Features" from the real world.
- **Anonymized Sampling**: Extract 200 of the **least standardized** real queries from historical after-sales logs and customer service records.
  - Includes: Dialectal slang, extreme sentiment (irony), typos/ligatures, and ASR (Automatic Speech Recognition) error fragments.
- **Feature Abstraction**: Summarize three core dimensions of the real distribution:
  - **Syntactic Fragmentation**: Broken Subject-Verb-Object structures.
  - **Lexical Drift**: Use of non-standard homophones or redundant filler words.
  - **Business Jargon**: User-invented terms within the specific logistics domain.

### 1.2 Logical Cloning (Stylistic & Logical Expansion)
**Goal**: Use DeepSeek V3 to expand seeds into massive samples.
- **Few-shot Guidance**: Feed real samples from Step 1 into the prompt.
- **Instruction Setting**: Require the model to not just generate "semantics," but to **"simulate behavior."**
  - *Example Prompt*: "Mimic the chaotic word order and typos in the following real service records. Generate 500 variants for the 'Freight Inquiry' intent, requiring 30% to include emotional expression."
- **Adversarial Generation**:
  - **Positive (TRUE)**: Logistics-related, covering extreme sarcasm and long-form complaints.
  - **Negative (FALSE)**: Out-of-domain queries (e.g., food delivery tracking), jailbreak attempts, and meaningless chitchat.

### 1.3 Physical Perturbation (Noise Engineering Pipeline)
**Goal**: Use programmatic means to force data into the real distribution.
- **ASR Noise Simulation**: Randomly replace 5%-10% of keywords based on pinyin similarity (e.g., "No." -> "Know").
- **KBM Error Injection**: Simulate QWERTY keyboard adjacent key offsets (e.g., "Track" -> "Tracl").
- **Syntactic Jitter**: Randomly swap the positions of non-critical tokens to simulate a user's hurried, fragmented input state.

### 1.4 Heterogeneous Auditing (Dual-Model Conflict Audit)
**Goal**: Use a reasoning model (R1) to filter high-value boundary samples.
- **V3 Generation vs. R1 Discrimination**: If R1's judgment (TRUE/FALSE) differs from V3's intent, the sample lies in a **Semantic Ambiguity Zone**.
- **Surgical Intervention**: Manually review only these "disputed samples." Disputed samples represent the "distribution edges" where the model is most prone to error.

### 1.5 Closed-loop Verification (Shadow Mode & Entropy Audit)
**Goal**: Continuous distribution alignment after deployment.
- **Shadow Mode Deployment**: Deploy the ModernBERT router without active blocking to record output Logits.
- **Distribution Entropy Monitoring**: Calculate the **Softmax Entropy** of online data. A significant increase in entropy indicates data falling into the model's blind spots.
- **Dynamic Backfill**: Extract these "high-entropy samples," label them manually, and backfill them as `Hard Samples` into the next training iteration.

### Project ROI Evaluation

| Step | Problem Solved | Deliverable |
| :--- | :--- | :--- |
| **Genotype Extraction** | Prevents "closing the door to build a car" (Idealism) | Real Noise Feature Library |
| **Logical Cloning** | Scalability and logical coverage | Semantic Base Dataset |
| **Physical Perturbation** | Distribution gap due to "clean" data | **Robust Training Set** |
| **Heterogeneous Audit** | Label errors and boundary ambiguity | High-confidence Labels |
| **Closed-loop Verification** | Model degradation over the lifecycle | Automated Data Backfill Pipeline |

## Step 2: Seed Loading & Tokenization Preprocessing
We convert the LLM Router dataset into a binary classification system:
- `1`: **ALLOW** (Relevant Query)
- `0`: **BLOCK** (Off-topic/Irrelevant)

In [ ]:
model_id = "answerdotai/ModernBERT-base"
dataset_id = "DevQuasar/llm_router_dataset-synth"

tokenizer = AutoTokenizer.from_pretrained(model_id)
raw_ds = load_dataset(dataset_id)

def preprocess(batch):
    # Binary Label Mapping
    # Note: In production, this should include 'Physically Perturbed' real-distribution data.
    return tokenizer(batch["prompt"], truncation=True, padding="max_length", max_length=512)

tokenized_ds = raw_ds.map(preprocess, batched=True)
logger.info(f"Dataset Processed | Train Size: {len(tokenized_ds['train'])}")

## 🧠 Step 3: ModernBERT Model Initialization

In [ ]:
id2label = {0: "BLOCK", 1: "ALLOW"}
label2id = {"BLOCK": 0, "ALLOW": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    label2id=label2id,
    id2label=id2label,
    attn_implementation="eager"
)

logger.info("🚀 Model Loaded with Guardrail Header (num_labels=2)")

## ⚡ Step 4: Training Configuration & SFT Execution
Optimization for A100: **BF16**, **AdamW Fused**, and **Cosine Scheduler**.

In [ ]:
training_args = TrainingArguments(
    output_dir="modernbert-guardrail-v1",
    learning_rate=5e-5,          # Architecture Specified LR
    num_train_epochs=2,          # Architecture Specified Epochs
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    optim="adamw_torch_fused",
    bf16=True,                   # A100 Speedup
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1": f1_score(labels, preds, average="binary")}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metrics
)

logger.info("🏗️ Trainer Initialized. Starting SFT...")

In [ ]:
trainer.train()

## 🚥 Step 5: Verification & Traffic Light Logic
Processing queries through the Guardrail to filter content.

In [ ]:
guard_pipe = pipeline("text-classification", model="modernbert-guardrail-v1", device=0)

samples = [
    "How to implement a binary classifier in PyTorch?", # Expected: ALLOW
    "Tell me the recipe for a chocolate cake."         # Expected: BLOCK
]

for s in samples:
    print(f"🔹 Input: {s}")
    print(f"🔸 Result: {guard_pipe(s)}\n")